In [ ]:
folder_path = '/Users/allisonpickle/Desktop/Gaussian Test'

In [ ]:
def Apply_Gaussian_Filters(input_folder, sigma_values=(1, 2, 3, 4, 5)):
    """
    Applies multiple Gaussian filters to all images in a folder.
    
    Creates separate output folders for each sigma value:
    - input_folder/sigma_1/
    - input_folder/sigma_2/
    - input_folder/sigma_3/
    - input_folder/sigma_4/
    - input_folder/sigma_5/
    
    Each folder contains all input images processed with that sigma value.
    
    Parameters:
    - input_folder: Path to folder containing images (TIFF, PNG, JPG supported)
    - sigma_values: Tuple of sigma values to test (default: 1, 2, 3, 4, 5)
    
    Returns:
    - Dictionary mapping sigma values to output folder paths
    """
    import os
    import numpy as np
    from skimage import io, filters
    from pathlib import Path
    
    print(f"Input folder: {input_folder}")
    print(f"Sigma values to test: {sigma_values}")
    print("=" * 80)
    
    # Find all image files
    supported_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(input_folder) 
                   if f.lower().endswith(supported_extensions)]
    image_files.sort()
    
    if not image_files:
        print("❌ No image files found!")
        return {}
    
    print(f"Found {len(image_files)} images to process\n")
    
    # Create output folders for each sigma
    output_folders = {}
    for sigma in sigma_values:
        folder_name = f"sigma_{sigma}"
        output_path = os.path.join(input_folder, folder_name)
        os.makedirs(output_path, exist_ok=True)
        output_folders[sigma] = output_path
        print(f"✓ Created output folder: {folder_name}/")
    
    print("\n" + "=" * 80)
    print("PROCESSING IMAGES")
    print("=" * 80 + "\n")
    
    # Process each image
    for idx, filename in enumerate(image_files, 1):
        print(f"[{idx}/{len(image_files)}] Processing: {filename}")
        
        # Load image
        input_path = os.path.join(input_folder, filename)
        try:
            image = io.imread(input_path)
        except Exception as e:
            print(f"  ✗ Error loading image: {e}")
            continue
        
        # Get base filename without extension
        base_name = os.path.splitext(filename)[0]
        
        # Apply each Gaussian filter
        for sigma in sigma_values:
            try:
                # Apply Gaussian filter
                if image.ndim == 2:
                    # Grayscale image
                    filtered = filters.gaussian(image, sigma=sigma, preserve_range=True)
                elif image.ndim == 3:
                    # Color image - apply to each channel
                    filtered = np.zeros_like(image)
                    for channel in range(image.shape[2]):
                        filtered[:, :, channel] = filters.gaussian(
                            image[:, :, channel], sigma=sigma, preserve_range=True
                        )
                else:
                    print(f"  ⚠️  Skipping sigma={sigma} - unexpected image dimensions: {image.ndim}")
                    continue
                
                # Convert back to original dtype
                filtered = filtered.astype(image.dtype)
                
                # Save filtered image
                output_filename = f"{base_name}_sigma_{sigma}.tiff"
                output_path = os.path.join(output_folders[sigma], output_filename)
                io.imsave(output_path, filtered, check_contrast=False)
                
                print(f"  ✓ Sigma {sigma}: saved to {folder_name}/{output_filename}")
                
            except Exception as e:
                print(f"  ✗ Error applying sigma={sigma}: {e}")
        
        print()  # Blank line between images
    
    # Summary
    print("=" * 80)
    print("PROCESSING COMPLETE")
    print("=" * 80)
    print(f"Total images processed: {len(image_files)}")
    print(f"Sigma values tested: {len(sigma_values)}")
    print(f"\nOutput folders created:")
    for sigma, folder_path in output_folders.items():
        num_files = len([f for f in os.listdir(folder_path) if f.endswith('.tiff')])
        print(f"  sigma_{sigma}/  ({num_files} images)")
    print("=" * 80 + "\n")
    
    return output_folders


# Example usage:
folder_path = '/Users/allisonpickle/Desktop/Gaussian Test'

output_folders = Apply_Gaussian_Filters(
    input_folder=folder_path,
    sigma_values=(1, 2, 3, 4, 5)
)

# If you want to test different sigma values:
# output_folders = Apply_Gaussian_Filters(
#     input_folder=folder_path,
#     sigma_values=(0.5, 1, 1.5, 2, 3, 5, 10)
# )

In [ ]:
def Threshold_Test(folder_path):
    import os
    from skimage import io
    from skimage.filters import try_all_threshold
    import matplotlib.pyplot as plt
    """Test threshold options for images in a folder

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the images

    Returns
    -------
    None
    """
    # Get a list of all image files in the folder
    image_files = [file for file in os.listdir(folder_path) if file.endswith('.tiff') or file.endswith('.tif')]

    # Iterate over each image file
    for image_file in image_files:
        # Construct the full path to the image file
        image_path = os.path.join(folder_path, image_file)

        # Read the image
        img = io.imread(image_path)
        fig, ax = try_all_threshold(img, figsize=(20, 18), verbose=False)
        plt.title(image_file)  # Add the image file name as the title
        plt.show()

# Example usage:
Threshold_Test('/Users/allisonpickle/Desktop/Gaussian Test/sigma_1')

Before use: Isolate microglia channel into a new folder for processing

Autocrop microglia of interest for morphology assessment

In [ ]:
def Crop_Microglia(image_path, output_folder, padding=20, min_size_microns=250, bbox_list=None):
    """Crops microglia from image. If bbox_list provided, uses those coordinates."""
    import cv2
    import os
    import numpy as np
    import random
    from skimage.measure import regionprops, label as skimage_label
    
    scale = 0.641  # microns per pixel
    min_size_pixels = int(min_size_microns / (scale ** 2))
    
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Error loading: {image_path}")
    
    # Detect or use provided bounding boxes
    if bbox_list is None:
        _, binary = cv2.threshold(cv2.blur(image, (5, 5)), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        props = [r for r in regionprops(skimage_label(binary)) if r.area >= min_size_pixels]
        bbox_list = random.sample(props, min(5, len(props))) if len(props) > 5 else props
        print(f"Detected {len(bbox_list)} objects")
    
    # Crop and save
    base_name = os.path.basename(image_path)
    h, w = image.shape
    cropped_count = 0
    
    for region in bbox_list:
        bbox = region.bbox if hasattr(region, 'bbox') else region
        r1, c1, r2, c2 = bbox
        
        # Skip edge-touching objects
        if r1 == 0 or c1 == 0 or r2 == h or c2 == w:
            continue
        
        # Apply padding
        r1, c1 = max(r1 - padding, 0), max(c1 - padding, 0)
        r2, c2 = min(r2 + padding, h), min(c2 + padding, w)
        
        cropped_count += 1
        new_name = base_name.replace("_C_0", f"_{cropped_count}_C_0")
        cv2.imwrite(os.path.join(output_folder, new_name), image[r1:r2, c1:c2])
    
    return [r.bbox for r in bbox_list] if bbox_list and hasattr(bbox_list[0], 'bbox') else None


def Isolate_Microglia(folder, padding=20, min_size_microns=250, bbox_dict=None):
    """Process all _C_0.tiff images in folder, grouping by base name."""
    import os
    from collections import defaultdict
    
    output_folder = os.path.join(folder, "Isolated microglia")
    os.makedirs(output_folder, exist_ok=True)
    
    # Group files by base name (removing _sigma_X)
    groups = defaultdict(list)
    for f in os.listdir(folder):
        if f.endswith("_C_0.tiff"):
            parts = f.split('_sigma_')
            base = f"{parts[0]}_{parts[1].split('_', 1)[1]}" if len(parts) == 2 else f
            groups[base].append(f)
    
    # Process each group
    local_bbox_dict = {}
    for base_name, files in groups.items():
        files.sort()
        print(f"\nGroup: {base_name} ({len(files)} files)")
        
        # Use existing bboxes or detect new ones
        if bbox_dict and base_name in bbox_dict:
            bbox_list = bbox_dict[base_name]
            print(f"Using existing bboxes")
        else:
            bbox_list = Crop_Microglia(os.path.join(folder, files[0]), output_folder, padding, min_size_microns)
            files = files[1:]
        
        # Apply to remaining files
        for f in files:
            Crop_Microglia(os.path.join(folder, f), output_folder, padding, min_size_microns, bbox_list)
        
        local_bbox_dict[base_name] = bbox_list
    
    return local_bbox_dict


def Process_All_Sigma_Folders(main_dir, padding=20, min_size_microns=250, ref_folder='sigma_3'):
    """Process all sigma_* folders, using ref_folder as reference for detecting microglia."""
    import os
    
    print(f"Processing: {main_dir}\nReference: {ref_folder}\n{'='*80}")
    
    # Find sigma folders with _C_0.tiff files
    folders = sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) 
                      if os.path.isdir(os.path.join(main_dir, f)) and f.startswith('sigma_')
                      and any(x.endswith('_C_0.tiff') for x in os.listdir(os.path.join(main_dir, f)))])
    
    if not folders:
        print("❌ No sigma_* folders found!")
        return
    
    # Find reference folder
    ref_path = next((f for f in folders if os.path.basename(f) == ref_folder), folders[0])
    folders = [ref_path] + [f for f in folders if f != ref_path]
    
    print(f"Processing order: {[os.path.basename(f) for f in folders]}\n{'='*80}")
    
    # Process folders
    master_bbox_dict = None
    for i, folder in enumerate(folders):
        print(f"\n[{i+1}/{len(folders)}] {os.path.basename(folder)}")
        try:
            if i == 0:
                master_bbox_dict = Isolate_Microglia(folder, padding, min_size_microns)
                print(f"✓ Detected {len(master_bbox_dict)} groups")
            else:
                Isolate_Microglia(folder, padding, min_size_microns, master_bbox_dict)
                print("✓ Applied reference bboxes")
        except Exception as e:
            print(f"✗ Error: {e}")
    
    print(f"\n{'='*80}\nCOMPLETE: {len(folders)} folders processed\n{'='*80}")


# Usage:
Process_All_Sigma_Folders('/Users/allisonpickle/Desktop/Gaussian Test')

In [ ]:
def sync_isolated_microglia_folders(main_directory, reference_folder_name='sigma_3', dry_run=True):
    """
    Synchronizes Isolated Microglia folders across all sigma_* subfolders to match
    the reference folder (default: sigma_3). Deletes images from other folders if they
    don't exist in the reference folder.
    
    Directory structure expected:
    main_directory/
    ├── sigma_1/
    │   └── Isolated Microglia/
    │       ├── image1_1_C_0.tiff
    │       └── image1_2_C_0.tiff
    ├── sigma_3/
    │   └── Isolated Microglia/
    │       └── image1_1_C_0.tiff  (image1_2 was deleted)
    └── ...
    
    Parameters:
    -----------
    main_directory : str
        Path to the main directory containing sigma_* subfolders
    reference_folder_name : str, optional
        Name of the reference folder to use as the standard (default: 'sigma_3')
    dry_run : bool, optional
        If True, only shows what would be deleted without actually deleting.
        If False, performs the actual deletion. (default: True for safety)
    
    Returns:
    --------
    summary : dict
        Dictionary containing deletion summary information
    """
    import os
    
    print(f"\n{'='*80}")
    print(f"SYNCHRONIZING ISOLATED MICROGLIA FOLDERS")
    print(f"{'='*80}\n")
    print(f"Main directory: {main_directory}")
    print(f"Reference folder: {reference_folder_name}")
    print(f"Mode: {'DRY RUN (preview only)' if dry_run else 'DELETION MODE (will delete files)'}")
    print()
    
    # Verify main directory exists
    if not os.path.exists(main_directory):
        print(f"❌ ERROR: Main directory not found: {main_directory}")
        return None
    
    # Find the reference folder
    reference_folder_path = None
    reference_isolated_path = None
    
    for item in os.listdir(main_directory):
        item_path = os.path.join(main_directory, item)
        if os.path.isdir(item_path) and item == reference_folder_name:
            reference_folder_path = item_path
            reference_isolated_path = os.path.join(item_path, 'Isolated Microglia')
            break
    
    if reference_folder_path is None:
        print(f"❌ ERROR: Reference folder '{reference_folder_name}' not found in {main_directory}")
        return None
    
    if not os.path.exists(reference_isolated_path):
        print(f"❌ ERROR: 'Isolated Microglia' folder not found in {reference_folder_path}")
        return None
    
    print(f"✓ Found reference folder: {reference_folder_path}")
    print(f"✓ Reference Isolated Microglia: {reference_isolated_path}")
    print()
    
    # Get list of images in reference folder
    try:
        reference_images = set(os.listdir(reference_isolated_path))
        reference_images = {f for f in reference_images 
                           if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))}
    except Exception as e:
        print(f"❌ ERROR: Unable to read reference folder: {e}")
        return None
    
    print(f"Reference folder contains {len(reference_images)} image(s)")
    print(f"{'─'*80}")
    print(f"Reference images:")
    for img in sorted(reference_images):
        print(f"  ✓ {img}")
    print(f"{'─'*80}\n")
    
    # Track statistics
    folders_processed = 0
    total_images_deleted = 0
    deletion_details = {}
    
    # Find all other sigma folders
    sigma_folders = []
    for item in os.listdir(main_directory):
        item_path = os.path.join(main_directory, item)
        if os.path.isdir(item_path) and item.startswith('sigma_') and item != reference_folder_name:
            isolated_path = os.path.join(item_path, 'Isolated Microglia')
            if os.path.exists(isolated_path):
                sigma_folders.append((item, item_path, isolated_path))
    
    if not sigma_folders:
        print("⚠️  No other sigma folders found to synchronize")
        return {
            'reference_folder': reference_folder_path,
            'reference_images': list(reference_images),
            'folders_processed': 0,
            'images_deleted': 0,
            'deletion_details': {}
        }
    
    print(f"Found {len(sigma_folders)} other sigma folder(s) to synchronize:")
    for folder_name, _, _ in sigma_folders:
        print(f"  - {folder_name}")
    print()
    
    # Process each folder
    for folder_name, folder_path, isolated_path in sorted(sigma_folders):
        print(f"{'='*80}")
        print(f"Processing: {folder_name}")
        print(f"{'='*80}")
        
        try:
            # Get list of images in this folder
            current_images = set(os.listdir(isolated_path))
            current_images = {f for f in current_images 
                            if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))}
            
            print(f"Current folder contains {len(current_images)} image(s)")
            
            # Find images that exist here but not in reference
            images_to_delete = current_images - reference_images
            
            if not images_to_delete:
                print(f"✓ Already synchronized - no images to delete")
                folders_processed += 1
            else:
                print(f"⚠️  Found {len(images_to_delete)} image(s) to delete:")
                
                deleted_images = []
                
                for img in sorted(images_to_delete):
                    img_path = os.path.join(isolated_path, img)
                    
                    if dry_run:
                        print(f"  [DRY RUN] Would delete: {img}")
                        deleted_images.append(img)
                    else:
                        try:
                            os.remove(img_path)
                            print(f"  ✗ Deleted: {img}")
                            deleted_images.append(img)
                            total_images_deleted += 1
                        except Exception as e:
                            print(f"  ❌ Error deleting {img}: {e}")
                
                deletion_details[folder_name] = deleted_images
                folders_processed += 1
            
            print()
            
        except Exception as e:
            print(f"❌ Error processing {folder_name}: {e}")
            import traceback
            traceback.print_exc()
            print()
    
    # Summary
    print(f"{'='*80}")
    print(f"SYNCHRONIZATION {'PREVIEW' if dry_run else 'COMPLETE'}")
    print(f"{'='*80}")
    print(f"Reference folder: {os.path.basename(reference_folder_path)}")
    print(f"Reference images: {len(reference_images)}")
    print(f"Folders processed: {folders_processed}")
    
    if dry_run:
        print(f"Images that WOULD BE deleted: {sum(len(imgs) for imgs in deletion_details.values())}")
    else:
        print(f"Images deleted: {total_images_deleted}")
    
    if deletion_details:
        print(f"\nDeletion breakdown:")
        for folder, images in sorted(deletion_details.items()):
            print(f"  {folder}: {len(images)} image(s)")
            for img in sorted(images)[:5]:
                print(f"    - {img}")
            if len(images) > 5:
                print(f"    ... and {len(images) - 5} more")
    else:
        print(f"\n✓ All folders already synchronized!")
    
    if dry_run:
        print(f"\n⚠️  DRY RUN MODE - No files were actually deleted")
        print(f"   Run with dry_run=False to perform actual deletion")
    
    print(f"{'='*80}\n")
    
    return {
        'reference_folder': reference_folder_path,
        'reference_images': list(reference_images),
        'folders_processed': folders_processed,
        'images_deleted': total_images_deleted if not dry_run else 0,
        'deletion_details': deletion_details,
        'dry_run': dry_run
    }


# Example usage:
summary = sync_isolated_microglia_folders(
    main_directory='/Users/allisonpickle/Desktop/Gaussian Test',
    reference_folder_name='sigma_0',
    dry_run=True  # Preview first
)

# Then run with dry_run=False to actually delete
summary = sync_isolated_microglia_folders(
    main_directory='/Users/allisonpickle/Desktop/Gaussian Test',
    reference_folder_name='sigma_3',
    dry_run=False
)

In [ ]:
def Measure_Microglia_Morphology_Single_Sigma_Optimized(colorized_image_path, 
                                                         selected_images=None, 
                                                         contour_sample=None,
                                                         contour_dpi=150):
    """
    Measure morphology of microglia in a single sigma folder with optimized contour saving.
    
    Parameters:
    - colorized_image_path: Path to the folder containing microglia images
    - selected_images: List of specific image filenames to process (if None, processes all)
    - contour_sample: Set of image filenames to save contours for (None = no contours)
    - contour_dpi: DPI for saved contour images (default: 150)
    
    Returns:
    - DataFrame with results including sigma information
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage import measure, io
    from skimage.filters import threshold_mean
    from skimage.morphology import binary_closing, disk, convex_hull_image
    from skimage.measure import regionprops, label, perimeter
    import pandas as pd
    import math
    from scipy.ndimage import center_of_mass

    def calculate_inertia_tensor(binary_image):
        """Calculate the inertia tensor of a binary image based on its center of mass."""
        com = center_of_mass(binary_image)
        y_indices, x_indices = np.indices(binary_image.shape)
        x_distances = x_indices - com[1]
        y_distances = y_indices - com[0]
        Ixx = np.sum(binary_image * y_distances**2)
        Iyy = np.sum(binary_image * x_distances**2)
        Ixy = -np.sum(binary_image * x_distances * y_distances)
        return np.array([[Ixx, Ixy], [Ixy, Iyy]])

    def calculate_aspect_ratio(eigenvalues):
        """Calculate the aspect ratio as the square root of the ratio of the largest to the smallest eigenvalue."""
        return np.sqrt(np.max(eigenvalues) / np.min(eigenvalues))

    def get_sigma_from_path(path):
        """Extract sigma from parent folder name (e.g., sigma_3 -> 3)"""
        parent = os.path.basename(os.path.dirname(path))
        if parent.startswith('sigma_'):
            return parent.split('_')[1]
        return 'Unknown'

    # Get sigma from folder path
    sigma = get_sigma_from_path(colorized_image_path)
    
    # Set scale for 20x Objective
    scale = 0.641  # microns per pixel

    # Create a "Contours" folder if we're saving any contours
    if contour_sample:
        contours_folder = os.path.join(colorized_image_path, "Contours")
        os.makedirs(contours_folder, exist_ok=True)

    # Get list of image files
    all_image_files = [file for file in os.listdir(colorized_image_path) 
                       if file.endswith('_C_0.tif') or file.endswith('_C_0.tiff')]
    
    # Use selected images if provided, otherwise use all
    if selected_images is not None:
        image_files = [f for f in all_image_files if f in selected_images]
    else:
        image_files = all_image_files

    # Process each image
    results = []
    for image_file in image_files:
        image_path = os.path.join(colorized_image_path, image_file)
        image = io.imread(image_path, as_gray=True)

        # Apply Mean thresholding
        thresh = threshold_mean(image)
        binary_image = image > thresh
        binary_image = binary_closing(binary_image, disk(3))

        # Label connected components
        labeled_image = label(binary_image)
        properties = regionprops(labeled_image)

        if len(properties) == 0:
            print(f"Warning: No objects found in {image_file}")
            continue

        # Find largest region
        largest_region = max(properties, key=lambda x: x.area)

        # Calculate convex hull
        convex_hull = convex_hull_image(labeled_image == largest_region.label)
        convex_hull_perimeter_pixels = perimeter(convex_hull)
        convex_hull_perimeter_microns = convex_hull_perimeter_pixels * scale
        convex_hull_area_pixels = np.sum(convex_hull)
        convex_hull_area_microns = convex_hull_area_pixels * (scale ** 2)

        # Calculate measurements
        area_pixels = largest_region.area
        perimeter_pixels = largest_region.perimeter
        area_microns = area_pixels * (scale ** 2)
        perimeter_microns = perimeter_pixels * scale

        # Calculate circularity
        circularity = (4 * math.pi * area_microns) / (perimeter_microns ** 2) if perimeter_microns > 0 else 0

        # Calculate aspect ratio
        binary_region = labeled_image == largest_region.label
        inertia_tensor = calculate_inertia_tensor(binary_region)
        eigenvalues, eigenvectors = np.linalg.eigh(inertia_tensor)
        aspect_ratio = calculate_aspect_ratio(eigenvalues)

        # Calculate roughness
        roughness = perimeter_microns / convex_hull_perimeter_microns if convex_hull_perimeter_microns > 0 else 0

        # Extract base name (remove sigma_X_ prefix for matching across sigmas)
        # Example: image1_sigma_3_1_C_0.tiff -> image1_1_C_0.tiff
        parts = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '').split('_sigma_')
        if len(parts) == 2:
            prefix = parts[0]
            suffix_parts = parts[1].split('_', 1)  # Split on first underscore after sigma number
            if len(suffix_parts) == 2:
                base_name = f"{prefix}_{suffix_parts[1]}_C_0"
            else:
                base_name = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '')
        else:
            base_name = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '')

        # Append results
        results.append({
            "File": image_file,
            "Base Name": base_name,
            "Sigma": sigma,
            "Area (µm²)": area_microns,
            "Perimeter (µm)": perimeter_microns,
            "Convex Hull Area (µm²)": convex_hull_area_microns,
            "Convex Hull Perimeter (µm)": convex_hull_perimeter_microns,
            "Circularity": circularity,
            "Aspect Ratio": aspect_ratio,
            "Roughness": roughness
        })

        # Save contour ONLY if in sample
        if contour_sample and image_file in contour_sample:
            fig, ax = plt.subplots(figsize=(10, 10))
            ax.imshow(image, cmap='gray')
            
            # Find and plot contours
            contours = measure.find_contours(labeled_image == largest_region.label, 0.5)
            for contour in contours:
                ax.plot(contour[:, 1], contour[:, 0], linewidth=2, color='red')
            
            # OPTIMIZED: Use scatter instead of loop for convex hull
            convex_hull_coords = np.argwhere(convex_hull)
            ax.scatter(convex_hull_coords[:, 1], convex_hull_coords[:, 0], 
                      s=1, c='green', marker='o', alpha=0.5)
            
            ax.axis('off')
            
            # Create output filename as PNG
            base_filename = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '')
            output_filename = f"{base_filename}_contour.png"
            output_path = os.path.join(contours_folder, output_filename)
            
            # Save as PNG explicitly
            plt.savefig(output_path, dpi=contour_dpi, bbox_inches='tight', pad_inches=0, format='png')
            plt.close(fig)
    
    return pd.DataFrame(results)


def Process_All_Sigma_All_Images(main_directory, save_contours_sample=20, contour_dpi=150):
    """
    Process ALL microglia images across all sigma_* folders (including sigma_0).
    
    Workflow:
    1. Find all sigma_* folders with "Isolated Microglia" subfolders
    2. Process ALL images in each folder (calculations for all)
    3. Save contours only for a random sample (default: 20 per sigma)
    4. Combine all results into a single Excel file with pivot tables (sigma as columns)
    
    Parameters:
    - main_directory: Path to directory containing sigma_* subfolders
    - save_contours_sample: Number of random images per sigma to save contours for (default: 20)
                           Set to None for all images, or 0 for no contours
    - contour_dpi: DPI for saved contour images (default: 150, use 300 for publication)
    
    Expected structure:
    main_directory/
    ├── sigma_0/Isolated Microglia/ (original, unfiltered)
    ├── sigma_1/Isolated Microglia/
    ├── sigma_2/Isolated Microglia/
    ├── sigma_3/Isolated Microglia/
    ├── sigma_4/Isolated Microglia/
    └── sigma_5/Isolated Microglia/
    
    Output contours:
    main_directory/
    ├── sigma_0/Isolated Microglia/Contours/
    ├── sigma_1/Isolated Microglia/Contours/
    │   ├── image1_sigma_1_1_contour.png
    │   └── ...
    """
    import os
    import pandas as pd
    import random
    
    print("=" * 80)
    print("PROCESSING ALL MICROGLIA ACROSS ALL SIGMA VALUES (INCLUDING σ=0)")
    print("=" * 80)
    print(f"Main directory: {main_directory}")
    if save_contours_sample:
        print(f"Contours will be saved for: {save_contours_sample} random images per sigma")
        print(f"Contour format: PNG with '_contour' suffix")
    else:
        print(f"Contours: DISABLED (calculations only)")
    print(f"Contour DPI: {contour_dpi}\n")
    
    # Define expected sigma values (INCLUDING SIGMA_0)
    expected_sigmas = ['0', '1', '2', '3', '4', '5']
    
    # Find all sigma folders with "Isolated Microglia" subfolder
    sigma_folders = {}
    
    for sigma in expected_sigmas:
        folder_name = f'sigma_{sigma}'
        folder_path = os.path.join(main_directory, folder_name)
        isolated_path = os.path.join(folder_path, 'Isolated Microglia')
        
        if os.path.exists(isolated_path):
            # Count images
            image_count = sum(1 for f in os.listdir(isolated_path) 
                            if f.endswith('_C_0.tif') or f.endswith('_C_0.tiff'))
            sigma_folders[sigma] = isolated_path
            label = "(original)" if sigma == '0' else ""
            print(f"✓ Found: {folder_name}/Isolated Microglia ({image_count} images) {label}")
        else:
            print(f"✗ Missing: {folder_name}/Isolated Microglia")
    
    if not sigma_folders:
        print("\n❌ No sigma folders found!")
        return
    
    print(f"\n✓ Found {len(sigma_folders)} sigma folder(s)")
    
    # Process each sigma folder
    print("\n" + "=" * 80)
    print("PROCESSING ALL FOLDERS")
    print("=" * 80)
    
    all_results = []
    
    for sigma in sorted(sigma_folders.keys()):
        folder_path = sigma_folders[sigma]
        
        label = " (original, unfiltered)" if sigma == '0' else ""
        print(f"\nProcessing sigma_{sigma}{label}...")
        
        # Get all images
        all_images = [f for f in os.listdir(folder_path) 
                     if f.endswith('_C_0.tif') or f.endswith('_C_0.tiff')]
        
        # Randomly select images for contour saving
        if save_contours_sample and save_contours_sample > 0:
            contour_sample = set(random.sample(all_images, 
                                              min(save_contours_sample, len(all_images))))
        else:
            contour_sample = None
        
        try:
            # Process with selective contour saving
            results_df = Measure_Microglia_Morphology_Single_Sigma_Optimized(
                folder_path,
                selected_images=None,  # Process ALL images
                contour_sample=contour_sample,
                contour_dpi=contour_dpi
            )
            all_results.append(results_df)
            
            if contour_sample:
                print(f"  ✓ Processed {len(results_df)} images ({len(contour_sample)} contours saved as PNG)")
            else:
                print(f"  ✓ Processed {len(results_df)} images (no contours)")
                
        except Exception as e:
            print(f"  ✗ Error: {str(e)}")
            import traceback
            traceback.print_exc()
    
    # Combine results
    if not all_results:
        print("\n❌ No results to combine!")
        return
    
    print("\n" + "=" * 80)
    print("COMBINING RESULTS AND CREATING EXCEL FILE")
    print("=" * 80)
    
    combined_df = pd.concat(all_results, ignore_index=True)
    
    print(f"\nTotal rows: {len(combined_df)}")
    print(f"Unique microglia: {combined_df['Base Name'].nunique()}")
    print(f"Sigma values processed: {sorted(combined_df['Sigma'].unique())}")
    
    # Count images per sigma
    print(f"\nImages per sigma:")
    for sigma in sorted(combined_df['Sigma'].unique()):
        count = len(combined_df[combined_df['Sigma'] == sigma])
        label = " (original)" if sigma == '0' else ""
        print(f"  sigma_{sigma}{label}: {count} images")
    
    # Create Excel file with multiple sheets
    excel_path = os.path.join(main_directory, "Combined_Morphology_Results_All_Sigma.xlsx")
    
    measurements = ['Area (µm²)', 'Perimeter (µm)', 'Convex Hull Area (µm²)', 
                   'Convex Hull Perimeter (µm)', 'Circularity', 'Aspect Ratio', 'Roughness']
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: All data with all columns (long format)
        combined_df_sorted = combined_df.sort_values(['Base Name', 'Sigma']).reset_index(drop=True)
        combined_df_sorted.to_excel(writer, sheet_name='All_Data', index=False)
        
        # Sheets 2-8: Pivot tables with sigma as columns (wide format)
        for measurement in measurements:
            # Create pivot table: Base Name as rows, Sigma as columns
            pivot_df = combined_df.pivot_table(
                index='Base Name',
                columns='Sigma',
                values=measurement,
                aggfunc='first'
            )
            
            # Reorder columns to match sigma order (INCLUDING SIGMA_0)
            sigma_order = ['0', '1', '2', '3', '4', '5']
            existing_sigmas = [s for s in sigma_order if s in pivot_df.columns]
            pivot_df = pivot_df[existing_sigmas]
            
            # Rename columns to include 'sigma_' prefix for clarity
            pivot_df.columns = [f'sigma_{col}' for col in pivot_df.columns]
            
            # Reset index to make Base Name a column
            pivot_df = pivot_df.reset_index()
            
            # Clean sheet name (Excel has 31 character limit)
            sheet_name = measurement.replace(' (µm²)', '').replace(' (µm)', '').replace(' ', '_')
            sheet_name = sheet_name[:31]
            
            # Write to Excel
            pivot_df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"\n✓ Results exported to: {excel_path}")
    print(f"\nExcel sheets created:")
    print(f"  1. All_Data (long format: all columns, {len(combined_df)} rows)")
    for i, measurement in enumerate(measurements, 2):
        sheet_name = measurement.replace(' (µm²)', '').replace(' (µm)', '').replace(' ', '_')[:31]
        unique_base_names = combined_df['Base Name'].nunique()
        print(f"  {i}. {sheet_name} (wide format: Base Name | sigma_0 | sigma_1 | sigma_2 | sigma_3 | sigma_4 | sigma_5, {unique_base_names} rows)")
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Total images processed: {len(combined_df)}")
    print(f"Total unique microglia (Base Names): {combined_df['Base Name'].nunique()}")
    print(f"Total sigma folders: {len(sigma_folders)} (including σ=0 original)")
    if save_contours_sample:
        print(f"Contours saved: {save_contours_sample} per sigma ({save_contours_sample * len(sigma_folders)} total)")
        print(f"Contour format: PNG files with '_contour' suffix")
        print(f"Contour DPI: {contour_dpi}")
    else:
        print(f"Contours: Not saved (calculations only)")
    print(f"Excel file: {os.path.basename(excel_path)}")
    print(f"Format: Wide format with sigma as columns (sigma_0, sigma_1, sigma_2, sigma_3, sigma_4, sigma_5)")
    print(f"Note: sigma_0 represents original unfiltered images")
    print("=" * 80 + "\n")
    
    return combined_df


# ================================================================================
# USAGE OPTIONS
# ================================================================================

# OPTION 1: Fast - 20 contours per sigma for validation
# Process_All_Sigma_All_Images(folder_path, save_contours_sample=20, contour_dpi=150)

# OPTION 2: Very fast - no contours, just calculations
# Process_All_Sigma_All_Images(folder_path, save_contours_sample=0)

# OPTION 3: Slow - all contours at medium quality
Process_All_Sigma_All_Images('/Users/allisonpickle/Desktop/Gaussian Test', save_contours_sample=265, contour_dpi=150)

# OPTION 4: Very slow - all contours at publication quality
# Process_All_Sigma_All_Images(folder_path, save_contours_sample=None, contour_dpi=300)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def create_composite_images_for_sigma(main_directory,
                                      sigmas=('0', '1', '2', '3', '4', '5'),
                                      image_extensions=('.tif', '.tiff', '.png', '.jpg'),
                                      figsize_per_image=(3, 3),
                                      dpi=150):
    """
    Iterate through sigma_0, sigma_1, sigma_2, sigma_3, sigma_4, sigma_5 subfolders in main_directory and
    compile images of the same base name into composite images with columns labeled by sigma.
    Creates composites for both:
      - Isolated Microglia: main_directory/sigma_#/Isolated Microglia
      - Contours: main_directory/sigma_#/Isolated Microglia/Contours
    Column labels are the sigma value (σ = #). The image file name is NOT drawn in the figure.
    Output files are saved under:
      - main_directory/Sigma_Comparisons/Isolated_Microglia
      - main_directory/Sigma_Comparisons/Contours
    """
    def collect_by_basename(folder_paths):
        # Map: base_name -> {sigma: path}
        by_name = defaultdict(dict)
        for sigma, folder in folder_paths.items():
            if not os.path.exists(folder):
                continue
            try:
                files = [f for f in os.listdir(folder)
                         if any(f.lower().endswith(ext) for ext in image_extensions)]
            except Exception:
                continue
            for f in files:
                base = os.path.splitext(f)[0]
                # Remove sigma_X_ prefix to get base name for matching
                # Example: image1_sigma_3_1_contour -> image1_1_contour
                parts = base.split('_sigma_')
                if len(parts) == 2:
                    prefix = parts[0]
                    suffix_parts = parts[1].split('_', 1)
                    if len(suffix_parts) == 2:
                        base = f"{prefix}_{suffix_parts[1]}"
                by_name[base][sigma] = os.path.join(folder, f)
        return by_name

    def ensure_output_dirs(root):
        out_root = os.path.join(root, 'Sigma_Comparisons')
        iso_out = os.path.join(out_root, 'Isolated_Microglia')
        con_out = os.path.join(out_root, 'Contours')
        os.makedirs(iso_out, exist_ok=True)
        os.makedirs(con_out, exist_ok=True)
        return iso_out, con_out

    def make_composites(by_name, label, output_dir):
        # Only include images present in ALL requested sigmas
        complete = {name: paths for name, paths in by_name.items() if len(paths) == len(sigmas)}
        print(f"{label}: {len(complete)} composite(s) to create (complete across sigmas)")
        count = 0
        for base_name, paths in sorted(complete.items()):
            ncols = len(sigmas)
            fig, axes = plt.subplots(1, ncols,
                                     figsize=(figsize_per_image[0] * ncols, figsize_per_image[1]))
            if ncols == 1:
                axes = [axes]
            # Render each sigma column
            for ax_idx, sigma in enumerate(sigmas):
                ax = axes[ax_idx]
                img_path = paths[sigma]
                try:
                    img = Image.open(img_path)
                    arr = np.array(img)
                    if arr.ndim == 2:
                        ax.imshow(arr, cmap='gray')
                    else:
                        ax.imshow(arr)
                except Exception as e:
                    ax.text(0.5, 0.5, f'Error\nσ={sigma}', ha='center', va='center', color='red')
                ax.set_title(f'σ = {sigma}', fontsize=12, fontweight='bold', pad=8)
                ax.axis('off')
            plt.subplots_adjust(wspace=0.05, hspace=0, left=0.02, right=0.98, top=0.88, bottom=0.02)
            # Save without drawing file name text in the figure
            safe_name = base_name
            out_path = os.path.join(output_dir, f"{safe_name}.png")
            plt.savefig(out_path, dpi=dpi, bbox_inches='tight', facecolor='white', edgecolor='none')
            plt.close(fig)
            count += 1
        return count

    # Prepare folder maps
    isolated_paths = {}
    contour_paths = {}
    for s in sigmas:
        isolated_paths[s] = os.path.join(main_directory, f"sigma_{s}", "Isolated Microglia")
        contour_paths[s] = os.path.join(main_directory, f"sigma_{s}", "Isolated Microglia", "Contours")

    iso_out, con_out = ensure_output_dirs(main_directory)

    print("Collecting Isolated Microglia images...")
    iso_by_name = collect_by_basename(isolated_paths)

    print("Collecting Contour images...")
    con_by_name = collect_by_basename(contour_paths)

    print("Creating composites (Isolated Microglia)...")
    iso_count = make_composites(iso_by_name, "Isolated Microglia", iso_out)

    print("Creating composites (Contours)...")
    con_count = make_composites(con_by_name, "Contours", con_out)

    print(f"Done. Isolated composites: {iso_count}, Contour composites: {con_count}")
    return {
        "isolated_composites": iso_count,
        "contour_composites": con_count,
        "output_directories": {
            "isolated": iso_out,
            "contours": con_out
        }
    }

# Example usage:
main_dir = '/Users/allisonpickle/Desktop/Gaussian Test'
create_composite_images_for_sigma(main_dir)

In [ ]:
def Analyze_Gaussian_Sigma_Stats(
    excel_path,
    sheet_name=None,
    output_dir=None,
    expected_sigmas=('0', '1', '2', '3', '4', '5'),
    reference_sigma='0',
    alpha=0.05,
    filter_treatment=None  # Optional: Filter for specific treatment in filename
):
    """
    Performs repeated measures ANOVA on Gaussian filtering (sigma) data from Excel.
    
    Expected Excel structure:
    Column 0: Base Name (microglia identifier)
    Subsequent columns: sigma_0, sigma_1, sigma_2, sigma_3, sigma_4, sigma_5
    
    Performs:
    - Repeated measures ANOVA across sigma values
    - Pairwise t-tests with Bonferroni correction
    - Generates plots and saves statistics
    - Plots individual microglia trajectories across sigma with significance markers
    
    Parameters:
    - excel_path: Path to Excel file
    - sheet_name: Sheet name (None = first sheet)
    - output_dir: Directory for outputs (None = same as Excel)
    - expected_sigmas: Tuple of sigma values (default: '0' to '5')
    - reference_sigma: Reference sigma for comparisons (default: '0' = original)
    - alpha: Significance level (default: 0.05)
    - filter_treatment: Only process rows with this string in Base Name (default: None = all)
    """
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    
    print(f"Reading Excel file: {excel_path}")
    if filter_treatment:
        print(f"Filtering for Base Names containing: '{filter_treatment}'")
    print("=" * 80)
    
    # Load Excel
    if sheet_name is None:
        df = pd.read_excel(excel_path)
        sheet_name = pd.ExcelFile(excel_path).sheet_names[0]
    else:
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
    
    print(f"Sheet: {sheet_name}")
    print(f"Total rows in Excel: {len(df)}")
    
    # Get headers
    headers = list(df.columns)
    basename_col = headers[0]
    
    # Find sigma columns
    sigma_columns = {}
    for i, col in enumerate(headers[1:], 1):
        col_name = str(col).strip().lower()
        if col_name.startswith('sigma_'):
            sigma = col_name[6:]  # Remove 'sigma_' prefix
            if sigma in expected_sigmas:
                sigma_columns[sigma] = col
    
    print(f"Found {len(sigma_columns)} sigma columns: {list(sigma_columns.keys())}")
    
    # Set output directory
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(excel_path), 
                                   os.path.splitext(os.path.basename(excel_path))[0] + "_Statistics")
    os.makedirs(output_dir, exist_ok=True)
    
    # Filter data if treatment specified
    if filter_treatment:
        df = df[df[basename_col].astype(str).str.contains(filter_treatment, na=False)]
        print(f"Filtered to {len(df)} rows containing '{filter_treatment}'")
    
    # Collect data for each sigma
    data_dict = {}
    for sigma, col in sigma_columns.items():
        values = df[col].dropna().values
        data_dict[sigma] = values
    
    n_microglia = len(df)
    print(f"\nTotal microglia to analyze: {n_microglia}")
    print("=" * 80)
    
    # Descriptive statistics
    print("\nDESCRIPTIVE STATISTICS:")
    print("-" * 80)
    
    stats_summary = []
    for sigma in expected_sigmas:
        if sigma in data_dict:
            values = np.array(data_dict[sigma])
            label = " (original)" if sigma == '0' else ""
            stats_summary.append({
                'Sigma': sigma,
                'Mean': np.mean(values),
                'SD': np.std(values, ddof=1),
                'Min': np.min(values),
                'Max': np.max(values)
            })
            print(f"σ = {sigma:>1}{label}: Mean = {np.mean(values):10.4f}, SD = {np.std(values, ddof=1):10.4f}, "
                  f"Min = {np.min(values):10.4f}, Max = {np.max(values):10.4f}")
    
    # Repeated measures ANOVA
    print("\n" + "=" * 80)
    print("STATISTICAL ANALYSIS RESULTS")
    print("=" * 80)
    
    # Prepare data for ANOVA
    sigma_list = [s for s in expected_sigmas if s in data_dict]
    
    # Perform repeated measures ANOVA
    F_stat, p_value = stats.f_oneway(*[data_dict[s] for s in sigma_list])
    
    print("Repeated Measures ANOVA:")
    print("-" * 80)
    df_between = len(sigma_list) - 1
    df_within = n_microglia * len(sigma_list) - len(sigma_list)
    print(f"F({df_between}, {df_within}) = {F_stat:.4f}")
    print(f"p-value: {p_value:.2e}")
    
    if p_value < 0.001:
        interpretation = "HIGHLY SIGNIFICANT (p < 0.001) ***"
    elif p_value < 0.01:
        interpretation = "SIGNIFICANT (p < 0.01) **"
    elif p_value < 0.05:
        interpretation = "SIGNIFICANT (p < 0.05) *"
    else:
        interpretation = "NOT SIGNIFICANT (p ≥ 0.05) ns"
    
    print(f"Interpretation: {interpretation}")
    
    # Pairwise t-tests with Bonferroni correction
    print(f"\nPairwise paired t-tests vs σ={reference_sigma} (reference):")
    n_comparisons = len(sigma_list) - 1
    bonferroni_alpha = alpha / n_comparisons
    print(f"Bonferroni correction applied for {n_comparisons} comparisons")
    print("-" * 80)
    
    pairwise_results = []
    ref_data = data_dict[reference_sigma]
    
    for sigma in sigma_list:
        if sigma == reference_sigma:
            continue
        
        test_data = data_dict[sigma]
        t_stat, p_val = stats.ttest_rel(ref_data, test_data)
        
        # Significance stars
        if p_val < 0.001:
            sig = "***"
        elif p_val < 0.01:
            sig = "**"
        elif p_val < bonferroni_alpha:
            sig = "*"
        else:
            sig = "ns"
        
        pairwise_results.append({
            'Comparison': f"σ={reference_sigma} vs σ={sigma}",
            'Sigma': sigma,
            't': t_stat,
            'p': p_val,
            'sig': sig
        })
        
        print(f"σ={reference_sigma} vs σ={sigma}: t = {t_stat:7.4f}, p = {p_val:.2e}")
        print(f"Significance: {sig}")
        print("-" * 80)
    
    print("\nSignificance levels: ns (p≥0.05), * (p<0.05), ** (p<0.01), *** (p<0.001)")
    print(f"(Note: Bonferroni correction applied for {n_comparisons} comparisons)")
    
    # Save statistics to file
    stats_file = os.path.join(output_dir, 
                               f"{os.path.splitext(os.path.basename(excel_path))[0]}_Statistics.txt")
    
    with open(stats_file, 'w') as f:
        f.write(f"Gaussian Filtering Analysis\n")
        f.write(f"Number of microglia: {n_microglia}\n")
        if filter_treatment:
            f.write(f"Treatment filter: {filter_treatment}\n")
        f.write("\nDESCRIPTIVE STATISTICS:\n")
        for s in stats_summary:
            label = " (original)" if s['Sigma'] == '0' else ""
            f.write(f"σ = {s['Sigma']}{label}: Mean={s['Mean']:10.4f}, SD={s['SD']:10.4f}, "
                   f"Min={s['Min']:10.4f}, Max={s['Max']:10.4f}\n")
        f.write(f"\nREPEATED MEASURES ANOVA:\n")
        f.write(f"F({df_between}, {df_within}) = {F_stat:.4f}, p = {p_value:.2e}\n")
        f.write(f"{interpretation}\n\n")
        f.write(f"PAIRWISE T-TESTS (Bonferroni corrected):\n")
        for pr in pairwise_results:
            f.write(f"{pr['Comparison']}: t={pr['t']:7.4f}, p={pr['p']:.2e}, {pr['sig']}\n")
    
    print(f"\nStatistics saved to {stats_file}")
    
    # Generate boxplot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    positions = list(range(len(sigma_list)))
    bp = ax.boxplot([data_dict[s] for s in sigma_list], positions=positions,
                    patch_artist=True, widths=0.6)
    
    for patch in bp['boxes']:
        patch.set_facecolor('#2196F3')
        patch.set_alpha(0.7)
    
    ax.set_xticks(positions)
    ax.set_xticklabels([f'σ={s}' for s in sigma_list])
    ax.set_xlabel('Gaussian Filter Sigma', fontsize=12)
    ax.set_ylabel(f'{sheet_name}', fontsize=12)
    title_text = f'Gaussian Filtering Comparison: {sheet_name}\n(n={n_microglia} microglia)'
    if filter_treatment:
        title_text += f' - {filter_treatment}'
    ax.set_title(title_text, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plot_file = os.path.join(output_dir,
                            f"{os.path.splitext(os.path.basename(excel_path))[0]}_sigma_comparison.png")
    plt.tight_layout()
    plt.savefig(plot_file, dpi=300)
    plt.close()
    
    print(f"Figure saved to {plot_file}")
    
    # Generate trajectory plot with significance markers
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # X-axis: sigma values as numbers
    x_values = [float(s) for s in sigma_list]
    
    # Plot each microglia as a separate line
    for i in range(n_microglia):
        y_values = [data_dict[s][i] for s in sigma_list]
        ax.plot(x_values, y_values, marker='o', alpha=0.3, linewidth=1, markersize=4, color='gray')
    
    # Add mean line
    mean_values = [np.mean(data_dict[s]) for s in sigma_list]
    ax.plot(x_values, mean_values, color='blue', linewidth=3, marker='s', 
            markersize=8, label='Mean', zorder=100)
    
    # Add significance asterisks above mean line
    y_max = max(mean_values)
    y_range = max([max(data_dict[s]) for s in sigma_list]) - min([min(data_dict[s]) for s in sigma_list])
    annotation_height = y_max + 0.1 * y_range
    
    # Create significance dictionary
    sig_dict = {pr['Sigma']: pr['sig'] for pr in pairwise_results}
    
    for i, sigma in enumerate(sigma_list):
        if sigma in sig_dict and sig_dict[sigma] != 'ns':
            ax.text(x_values[i], annotation_height, sig_dict[sigma], 
                   ha='center', va='bottom', fontsize=14, fontweight='bold', color='black')
    
    ax.set_xlabel('Gaussian Filter Sigma (σ)', fontsize=12)
    ax.set_ylabel(f'{sheet_name}', fontsize=12)
    title_text = f'Individual Microglia Trajectories: {sheet_name}\n(n={n_microglia} microglia)'
    if filter_treatment:
        title_text += f' - {filter_treatment}'
    ax.set_title(title_text, fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Add text annotation for reference
    ax.text(0.98, 0.02, f'Significance vs σ={reference_sigma}\n* p<0.05, ** p<0.01, *** p<0.001', 
           transform=ax.transAxes, ha='right', va='bottom', 
           fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    trajectory_file = os.path.join(output_dir,
                                  f"{os.path.splitext(os.path.basename(excel_path))[0]}_trajectories.png")
    plt.tight_layout()
    plt.savefig(trajectory_file, dpi=300)
    plt.close()
    
    print(f"Trajectory plot saved to {trajectory_file}")
    
    print(f"\n{'=' * 80}")
    print("ANALYSIS COMPLETE")
    print(f"All outputs saved to: {output_dir}")
    print("=" * 80)

# Example usage:
Analyze_Gaussian_Sigma_Stats(
    excel_path='/Users/allisonpickle/Desktop/Gaussian Test/Combined_Morphology_Results_All_Sigma.xlsx',
    sheet_name='Area',  # Choose: Area, Perimeter, Circularity, Aspect_Ratio, Roughness, etc.
    filter_treatment=None  # Optional: filter by treatment name in Base Name column
)

In [ ]:
import os
import pandas as pd

def combine_sigma_to_single_file(
    input_excel_path,
    sheet_name=None,
    basename_col=0,
    overwrite=False,
    expected_sigmas=('0','1','2','3','4','5'),
    output_format='csv'
):
    """
    Read an Excel sheet with multiple Gaussian sigma measurements.
    
    Input format:
      - Column 0: Filename (e.g., 240623_Tri_5ugLPS_2_1.tiff)
      - Columns 1-N: Sigma measurements (sigma_0, sigma_1, ..., sigma_5)
    
    Output: Single CSV with columns:
      Filename | Treatment | Biological_Rep | Well | Image | Sigma | Results

    Filename format:
      bioRep_wellID_treatment_imageNum.tiff
      Example: 240623_Tri_5ugLPS_2_1.tiff
      - parts[0] = biological replicate date (240623, 240624, 240625 → Bio Rep 1, 2, 3)
      - parts[2] = treatment (Ctrl, 5ugLPS, etc.)
      - Every 4 images = 1 Well (resets per treatment & sigma)
    """
    # Load sheet
    if sheet_name is None:
        df = pd.read_excel(input_excel_path)
        sheet_name = pd.ExcelFile(input_excel_path).sheet_names[0]
    else:
        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    headers = list(df.columns)

    # Identify sigma columns
    sigma_cols = []
    for i, col in enumerate(headers):
        if i == basename_col:
            continue
        name = str(col).strip().lower()
        if name.startswith('sigma_') or name.startswith('sigma'):
            sigma_val = name.replace('sigma_', '').replace('sigma', '')
            sigma_cols.append((sigma_val, col))
    
    # Fallback: use expected sigma order if columns aren't labeled
    if not sigma_cols:
        subsequent_cols = [c for idx, c in enumerate(headers) if idx != basename_col]
        for i, s in enumerate(expected_sigmas):
            if i < len(subsequent_cols):
                sigma_cols.append((s, subsequent_cols[i]))

    print(f"✓ Found {len(sigma_cols)} sigma columns: {[s[0] for s in sigma_cols]}")

    # Build biological replicate mapping from unique date tokens
    first_tokens = []
    seen = set()
    for idx in range(len(df)):
        name = df.iloc[idx, basename_col]
        if pd.isna(name):
            continue
        base = str(name).strip()
        parts = base.split('_')
        bio_token = parts[0] if len(parts) > 0 else ""
        if bio_token and bio_token not in seen:
            seen.add(bio_token)
            first_tokens.append(bio_token)
    token_to_rep = {tok: i+1 for i, tok in enumerate(first_tokens)}

    print(f"✓ Biological replicate mapping: {token_to_rep}")

    # Parse all rows for all sigma values
    all_rows = []
    
    for sigma_value, sigma_col in sigma_cols:
        for idx in range(len(df)):
            name = df.iloc[idx, basename_col]
            if pd.isna(name):
                continue
            name = str(name).strip()
            parts = name.split('_')

            # Parse filename components
            bio_token = parts[0] if len(parts) > 0 else ""
            treatment = parts[2] if len(parts) > 2 else ""

            all_rows.append({
                "Filename": name,
                "Treatment": treatment,
                "Biological_Rep": token_to_rep.get(bio_token),
                "Sigma": sigma_value,
                "Results": df.iloc[idx][sigma_col],
                "Original_Order": idx,
            })
    
    # Create combined dataframe
    combined_df = pd.DataFrame(all_rows)
    
    # Sort to maintain structure
    combined_df = combined_df.sort_values(["Sigma", "Original_Order"]).reset_index(drop=True)
    
    # Assign Well and Image: every 4 images = 1 Well (resets per treatment & sigma)
    combined_df["Well"] = 0
    combined_df["Image"] = 0
    
    for sigma in combined_df["Sigma"].unique():
        for treatment in combined_df["Treatment"].unique():
            mask = (combined_df["Sigma"] == sigma) & (combined_df["Treatment"] == treatment)
            treatment_rows = combined_df[mask]
            
            well_counter = 1
            for i, idx in enumerate(treatment_rows.index):
                combined_df.loc[idx, "Well"] = well_counter
                combined_df.loc[idx, "Image"] = ((i % 4) + 1)
                
                # Increment well every 4 images
                if (i + 1) % 4 == 0:
                    well_counter += 1
    
    # Final column selection
    combined_df = combined_df[["Filename", "Treatment", "Biological_Rep", "Well", "Image", "Sigma", "Results"]]
    
    # Build output path
    in_dir = os.path.dirname(os.path.abspath(input_excel_path))
    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    base_name = f"{in_stem}_Combined_Sigma"
    
    if output_format.lower() == 'csv':
        out_path = os.path.join(in_dir, f"{base_name}_RMixedFormatted.csv")
    else:
        out_path = os.path.join(in_dir, f"{base_name}_RMixedFormatted.xlsx")

    # Handle existing files
    if not overwrite and os.path.exists(out_path):
        base, ext = os.path.splitext(out_path)
        i = 1
        while True:
            candidate = f"{base} ({i}){ext}"
            if not os.path.exists(candidate):
                out_path = candidate
                break
            i += 1

    # Write file
    if output_format.lower() == 'csv':
        combined_df.to_csv(out_path, index=False)
    else:
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            combined_df.to_excel(writer, sheet_name="Data", index=False)

    # Summary
    print(f"\n{'='*60}")
    print(f"✓ COMBINED FILE CREATED: {out_path}")
    print(f"  Total rows: {len(combined_df)}")
    print(f"  Sigma values: {sorted(combined_df['Sigma'].unique())}")
    print(f"  Treatments: {sorted(combined_df['Treatment'].unique())}")
    print(f"  Biological replicates: {sorted(combined_df['Biological_Rep'].unique())}")
    print(f"  Wells per treatment×sigma: {combined_df.groupby(['Treatment', 'Sigma'])['Well'].max().to_dict()}")
    print(f"{'='*60}\n")
    
    return out_path


# ================================================================================
# USAGE
# ================================================================================
output_file = combine_sigma_to_single_file(
    input_excel_path='/Users/allisonpickle/Desktop/Gaussian Microglia.xlsx',
    sheet_name='Sheet1',
    output_format='csv',
    expected_sigmas=('0', '1', '2', '3', '4', '5')
)

print(f"Ready for R analysis! Feed this file to your mixed model script:")
print(f"  {output_file}")